# Basic Usage on UDA Benchmark Suite

The demonstration encompasses several essential steps:

* Prepare the question-answer-document triplet data-item
* Extract and segment the document content
* Build indexes and retrieve data segments
* Generate answering response with LLMs (**Together AI — Nemotron**)
* Evaluate the accuracy of responses using the specific metric.

## 0. Install dependencies

Run this once if you haven't already.

In [1]:
# !pip install together langchain chromadb sentence-transformers PyPDF2

## 1. Load and view the Q&A labels

The Q&A labels are in `dataset/qa/`. Set `DATASET_NAME` to:
- `"fin"` — FinHybrid (financial reports, Exact Match metric)
- `"tat"` — TatHybrid (finance, numeracy-aware F1)
- `"paper"` / `"nq"` — academic / Wikipedia (span F1)

In [2]:
import pandas as pd
from uda.utils import preprocess

DATASET_NAME = "fin"  # ← change to 'tat', 'paper', 'nq' etc.

csv_file_path = f"./dataset/qa/{DATASET_NAME}_qa.csv"
df = pd.read_csv(csv_file_path, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict = preprocess.qa_df_to_dict(DATASET_NAME, df)

# Preview
for key in list(qas_dict.keys())[:2]:
    print("Document Name: ", key)
    print("Its Q&A pairs: ", qas_dict[key][:2])
    print("=========================================")

Document Name:  ADI_2009
Its Q&A pairs:  [{'question': 'what is the the interest expense in 2009?', 'answers': {'str_answer': '380', 'exe_answer': '3.8'}, 'q_uid': 'ADI/2009/page_49.pdf-1'}, {'question': 'what is the expected growth rate in amortization expense in 2010?', 'answers': {'str_answer': '-27.0%', 'exe_answer': '-0.26689'}, 'q_uid': 'ADI/2009/page_59.pdf-2'}]
Document Name:  ABMD_2012
Its Q&A pairs:  [{'question': 'during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?', 'answers': {'str_answer': '', 'exe_answer': 'yes'}, 'q_uid': 'ABMD/2012/page_75.pdf-1'}, {'question': 'for equity awards where the performance criteria has been met in 2012 , what is the average compensation expense per year over which the cost will be expensed?', 'answers': {'str_answer': '1719526', 'exe_answer': '1714285.71429'}, 'q_uid': 'ABMD/2012/page_75.pdf-2'}]


## 2. Prepare the document data

`get_example_pdf_path()` looks in the small example folder that ships with the repo.  
Once you've downloaded the full dataset, swap it for `get_pdf_path()`.

In [3]:
example_doc_name = list(qas_dict.keys())[0]
example_qa = qas_dict[example_doc_name][1]

pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, example_doc_name)
# pdf_path = preprocess.get_pdf_path(DATASET_NAME, example_doc_name)  # full dataset

if pdf_path is None:
    print("No PDF found for this document — check dataset/src_doc_files_example/")
else:
    print(f"PDF path: {pdf_path}")

PDF path: dataset/src_doc_files_example/fin_docs/ADI_2009.pdf


### 2a. Extract raw text from the PDF

PyPDF2 extracts text page-by-page. Tables become flat text with spaces/newlines as structure.

In [4]:
import PyPDF2

pdf_text = ""
with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file, strict=False)
    for page_num in range(len(reader.pages)):
        page = reader.pages[page_num]
        pdf_text += page.extract_text()

print(f"Total characters extracted: {len(pdf_text):,}")
print("\nSample (chars 8000-8500):")
print(pdf_text[8000:8500])

Total characters extracted: 379,301

Sample (chars 8000-8500):
signiﬁcantly decreasing their
inventories. In response to these
unprecedented revenue declines, we
substantially decreased production levels to
reduce our inventory levels. This action, of
course, had the effect of temporarily lowering
our gross margins, which reached a trough of
54.1% in the third quarter.  We reacted
quickly to the business environment, reducingPresident’s Letter
0.000.100.200.300.400.500.60FY2009 Product Revenue and Diluted EPS
From Continuing Operations by Quarter
$0$100$200


### 2b. Segment into overlapping chunks

3000-character chunks with 10% overlap (300 chars). **Sprint 3 experiment:** try `chunk_size=1500` and `6000` to measure the effect on accuracy.

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,   # ← vary this: 1500 / 3000 / 6000
    chunk_overlap=300,
)
text_chunks = text_splitter.split_text(pdf_text)

print(f"Number of chunks: {len(text_chunks)}")
avg_words = sum(len(c.split()) for c in text_chunks) / len(text_chunks)
print(f"Avg words per chunk: {avg_words:.0f}")

/Users/jon/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Number of chunks: 141
Avg words per chunk: 437


## 3. Build vector index and retrieve relevant chunks

Embed chunks with `all-MiniLM-L6-v2`, store in ChromaDB, then query by cosine similarity. `top_k=5` is the default — try 3 and 10 for your experiment.

In [6]:
import importlib.util, os

# Load access_config.py directly by file path — avoids uda.utils import issues
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print("access_config loaded — model:", access_config.TOGETHER_MODEL)

access_config loaded — model: nvidia/nemotron-3-ultra-550b-a55b


In [7]:
import chromadb
from together import Together
# access_config already loaded above

# ── Together AI embedding function ────────────────────────────────────────
together_client = Together(api_key=access_config.TOGETHER_API_KEY)

class TogetherEmbeddingFunction:
    """Wraps Together AI embeddings in the interface ChromaDB expects."""
    def name(self):
        return "together-m2-bert"

    def __call__(self, input):
        response = together_client.embeddings.create(
            model="togethercomputer/m2-bert-80M-8k-retrieval",
            input=input,
        )
        return [r.embedding for r in response.data]

# ── Build ChromaDB collection ──────────────────────────────────────────────
# Delete if exists so re-running cells never causes conflicts
ef = TogetherEmbeddingFunction()
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("demo_vdb")
except Exception:
    pass  # didn't exist yet, fine

collection = chroma_client.create_collection(
    "demo_vdb",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)
id_list = [str(i) for i in range(len(text_chunks))]
collection.add(documents=text_chunks, ids=id_list)
print(f"Indexed {len(text_chunks)} chunks into ChromaDB via Together AI embeddings")

BadRequestError: Error code: 400 - {'id': 'opt2GKf-2kFHot-a1269df45b2e31bc', 'error': {'message': 'Unable to access non-serverless model togethercomputer/m2-bert-80M-8k-retrieval. Please visit https://api.together.ai/models/togethercomputer/m2-bert-80M-8k-retrieval to create and start a new dedicated endpoint for the model.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_available'}} in add.

In [ ]:
top_k = 5  # ← vary this: 3 / 5 / 10
question = example_qa["question"]

fetch_res = collection.query(query_texts=[question], n_results=top_k)
contexts = fetch_res["documents"][0]

print(f"Question: {question}\n")
for idx, context in enumerate(contexts):
    print(f"===== Context {idx+1} =======")
    print(context[:200], "...\n")

## 4. Generate an answer with Together AI (Nemotron)

This replaces the original Azure OpenAI call. The Together AI client uses the same OpenAI-compatible `.chat.completions.create()` interface, so the prompt structure is identical.

> **Before running:** open `uda/utils/access_config.py` and set `TOGETHER_API_KEY` to your key.

In [ ]:
from together import Together
from uda.utils import llm  # access_config already loaded above

# ── Together AI client ────────────────────────────────────────────────────
client = Together(api_key=access_config.TOGETHER_API_KEY)

# ── Build the prompt (same UDA helper, works with any backend) ────────────
# llm_type controls the prompt template:
#   'gpt-4'     → Chain-of-Thought template (used for fin / tat)
#   'llama-8B'  → instruction template for open-source models
llm_type = "gpt-4"  # keep this — it selects the CoT prompt format for finance

context_text = "\n".join(contexts)
llm_message = llm.make_prompt(
    question=question,
    context=context_text,
    task_name=DATASET_NAME,
    llm_type=llm_type,
)

# ── Call Together AI ──────────────────────────────────────────────────────
raw_response = client.chat.completions.create(
    model=access_config.TOGETHER_MODEL,
    messages=llm_message,
    temperature=0.1,
    max_tokens=512,
)

together_response = raw_response.choices[0].message.content

print(f"Question:        {question}")
print(f"Ground truth:    {example_qa['answers']}")
print(f"Model response:  {together_response}")

## 5. Evaluate the response

`eval_main()` automatically picks the right metric based on `DATASET_NAME`:
- `fin` → Exact Match with ±1% numerical tolerance
- `tat` → Numeracy-aware F1
- `paper` / `nq` / `feta` → Span-level F1

In [ ]:
from uda.eval.my_eval import eval_main

res_dict = {
    "question": question,
    "response": together_response,
    "doc": example_doc_name,
    "q_uid": example_qa["q_uid"],
    "answers": example_qa["answers"],
}
res_data = [res_dict]

print("Result dict:", res_data)
print()
eval_main(DATASET_NAME, res_data)

## 6. Sprint 3 — Scale up: loop over all Q&A pairs in a document

The cells above run on one question. This scaffold loops over every Q&A pair for a single document and collects results for batch evaluation. Extend the outer loop over `qas_dict.keys()` to run the full dataset.

In [ ]:
import time

# Pick a document
target_doc = list(qas_dict.keys())[0]
doc_qas = qas_dict[target_doc]

# Re-index the PDF (reuse collection from above if same doc, else rebuild)
# collection already built above for this doc — skip re-indexing

all_results = []

for qa in doc_qas:
    q = qa["question"]

    # Retrieve
    fetch = collection.query(query_texts=[q], n_results=top_k)
    ctx = "\n".join(fetch["documents"][0])

    # Prompt
    msg = llm.make_prompt(question=q, context=ctx,
                          task_name=DATASET_NAME, llm_type="gpt-4")

    # Infer
    resp = client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=msg,
        temperature=0.1,
        max_tokens=512,
    )
    answer = resp.choices[0].message.content

    all_results.append({
        "question": q,
        "response": answer,
        "doc": target_doc,
        "q_uid": qa["q_uid"],
        "answers": qa["answers"],
    })
    time.sleep(0.3)  # be polite to the API

print(f"Ran {len(all_results)} questions on document: {target_doc}")
eval_main(DATASET_NAME, all_results)